# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) provided at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Install mlcroissant if not previously installed
!pip install mlcroissant

## 1. Data Loading
Let's load the dataset metadata and review the main description using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

# Optionally pretty-print a few more metadata fields
print("Citation:")
print(metadata.citeAs)
print("\nLicense:")
print(metadata.license)


## 2. Data Overview
We'll enumerate the available record sets in the dataset and print their `@id`, `name`, and the list of included fields. This helps identify what data can be loaded and how to reference each part by `@id`.

Record sets represent structured tables or resources (e.g., survey responses, participant info). Each record set may have fields, which in turn map to columns or attributes.

In [ ]:
# List all record sets and their fields
print("Available Record Sets:")
record_sets = []

# The dataset's record sets can be accessed via dataset.metadata.record_sets
for rs in dataset.metadata.record_sets:
    record_sets.append(rs['@id'])
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(no name)')}")
    print(f"  description: {rs.get('description', '')}")
    field_ids = [field['@id'] for field in rs.get('fields', [])]
    print(f"  Fields (@id): {field_ids}\n")
if not record_sets:
    print("No record sets declared directly in .recordSet -- inspecting distributions.")

# If the recordSet list is empty (as in this dataset's root), try dataset.metadata.distributions
distribution_ids = []
for dist in dataset.metadata.distributions:
    distribution_ids.append(dist['@id'])
    print(f"Distribution @id: {dist['@id']}")
    print(f"  name: {dist.get('name', '(no name)')}")
    print(f"  encodingFormat: {dist.get('encodingFormat', '')}")
    print(f"  contentUrl: {dist.get('contentUrl', '')}\n")
    # Often, distributions are directly tied to record sets, or ARE the record sets (CSV files)

# For this dataset, let's auto-discover any record_sets programmatically (mlcroissant auto-scans Croissant structure)
if hasattr(dataset, 'record_set_ids'):
    detected_record_sets = list(dataset.record_set_ids)
    print("Detected record sets (by @id):")
    pprint.pprint(detected_record_sets, compact=True)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. **Always reference by the `@id` of the record set and fields/columns, as shown above.**

In [ ]:
# We'll auto-detect all record sets present
all_record_set_ids = list(dataset.record_set_ids)
print(f"All record sets (@id): {all_record_set_ids}")

# For demonstration, extract all records from the main record set (usually first/main survey table)
dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records. Columns (@id):")
    print(dataframes[record_set_id].columns.tolist())

# Let the user inspect the first table
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if main_record_set_id:
    print(f"---\nFirst five rows of {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We now demonstrate some transformations and simple analysis on a numeric field in the main record set. All field/column references are by `@id` as required for full reproducibility.

This block demonstrates:
- Filtering by a threshold on a numeric field
- Normalization
- Grouped aggregation by a categorical field

**Modify variables below based on actual columns you want to analyze from the printed list above.**

In [ ]:
# Select a numeric field and group field by their @ids
# Replace these with actual @id values from your dataframes
numeric_field = 'gad7_score'  # e.g., '@id' for the GAD-7 total score column
group_field = 'village'       # e.g., '@id' for a categorical field like participant's village
record_set_id = main_record_set_id

# Check if these fields exist (if not, adjust above accordingly after checking the printout in section 3)
if record_set_id and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    
    # Remove rows with NaN in the chosen field (typical in survey data)
    fdf = df.dropna(subset=[numeric_field]).copy()
    
    threshold = 10
    filtered_df = fdf[fdf[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optional group-by
    if group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGroup-mean {numeric_field} by {group_field}:")
        print(grouped.head())
else:
    print("Selected numeric field or group field not present. Update 'numeric_field'/'group_field' based on your actual columns.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (`gad7_score`) and its relationship to the group field (`village`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # Violin plot by group, if available
    if group_field in df.columns:
        plt.figure(figsize=(12,5))
        sns.violinplot(x=group_field, y=numeric_field, data=df, cut=0)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization fields not present. Set 'numeric_field'/'group_field' based on actual columns.")

## 6. Conclusion
In this notebook, we've loaded the Kilifi County Mental Health Survey data via Croissant, explored available record sets and their fields, inspected the main records, and performed initial filtering, normalization, and group-based analyses on selected variables. The `mlcroissant` library makes it straightforward to audit and process FAIR datasets using reproducible references such as resource and field `@id`s.

**Suggestions for further analysis:**
- Investigate relationships between multiple psychological scores (e.g., `gad7_score`, `phq9_score`).
- Stratify by demographic fields such as `age` or `level_of_education` (referenced by their `@id` values).
- Apply more advanced cleaning or missing data imputation, if needed.

For more, see https://github.com/mlcommons/croissant for Croissant standard documentation.